In [1]:
"""Utilities for Quad4FHE plaintext experiments.

This module is intentionally small and dependency-light so it can be copied next
into any single-dataset notebook/script. It provides two things:

1. A tee context manager that writes all notebook/script stdout/stderr to a log
   file while still showing it on screen.
2. Direct JSON export from quad4fhe.ReplacementResult objects, avoiding fragile
   regex parsing of copied notebook output.
"""

from __future__ import annotations

import contextlib
import json
import math
import sys
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, Iterable, Optional


class _TeeStream:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

    def isatty(self):
        return any(getattr(stream, "isatty", lambda: False)() for stream in self.streams)


@contextlib.contextmanager
def tee_stdout_stderr(log_path: str | Path):
    """Duplicate stdout/stderr to ``log_path`` and the current console.

    Use around the whole experiment:

        with tee_stdout_stderr("results/my_run.txt"):
            main()
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    old_stdout, old_stderr = sys.stdout, sys.stderr
    with log_path.open("w", encoding="utf-8", buffering=1) as fh:
        sys.stdout = _TeeStream(old_stdout, fh)
        sys.stderr = _TeeStream(old_stderr, fh)
        try:
            print(f"[autosave] stdout/stderr log -> {log_path}")
            yield log_path
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def to_jsonable(obj: Any) -> Any:
    """Convert common scientific Python objects to strict JSON values."""
    # Local imports keep this helper usable even in minimal environments.
    try:
        import numpy as np
    except Exception:  # pragma: no cover
        np = None
    try:
        import pandas as pd
    except Exception:  # pragma: no cover
        pd = None
    try:
        import torch
    except Exception:  # pragma: no cover
        torch = None

    if obj is None or isinstance(obj, (str, bool, int)):
        return obj

    if isinstance(obj, float):
        return obj if math.isfinite(obj) else None

    if np is not None:
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            value = float(obj)
            return value if math.isfinite(value) else None
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return to_jsonable(obj.tolist())

    if torch is not None and hasattr(torch, "is_tensor") and torch.is_tensor(obj):
        return to_jsonable(obj.detach().cpu().numpy())

    if isinstance(obj, Path):
        return str(obj)

    if pd is not None:
        if isinstance(obj, pd.DataFrame):
            return to_jsonable(obj.to_dict(orient="records"))
        if isinstance(obj, pd.Series):
            return to_jsonable(obj.to_dict())

    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]

    # Last resort: preserve readable values without breaking json.dump.
    return str(obj)


def save_json(obj: Dict[str, Any], path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as fh:
        json.dump(to_jsonable(obj), fh, indent=2, ensure_ascii=False, allow_nan=False)
    print(f"[autosave] JSON -> {path}")
    return path


def save_csv(rows: Iterable[Dict[str, Any]], path: str | Path) -> Path:
    import pandas as pd
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(list(rows)).to_csv(path, index=False)
    print(f"[autosave] CSV -> {path}")
    return path


def dataframe_records(df: Any) -> list:
    if df is None:
        return []
    try:
        return to_jsonable(df.to_dict(orient="records"))
    except Exception:
        return []


def dataframe_test_by_method(df: Any) -> Dict[str, Dict[str, Any]]:
    """Return {method: metrics} for rows with Split == 'test'."""
    out: Dict[str, Dict[str, Any]] = {}
    if df is None:
        return out
    try:
        sub = df[df["Split"] == "test"]
        for _, row in sub.iterrows():
            method = str(row.get("Method"))
            metrics = {str(k): to_jsonable(v) for k, v in row.to_dict().items()
                       if k not in ("Method", "Split")}
            out[method] = metrics
    except Exception:
        return out
    return out


def _metric_from_table(table: Dict[str, Dict[str, Any]], method: str, *keys: str) -> Any:
    row = table.get(method, {})
    for key in keys:
        if key in row:
            return row[key]
    return None


def quad_report_diagnostics(result: Any, split: str, n_expected: Optional[int] = None) -> Dict[str, Any]:
    """Extract agreement/mismatch diagnostics for fit/calibration or test split."""
    attr_candidates = []
    if split in ("fit", "calib", "calibration"):
        attr_candidates = ["fit_diagnostics", "calib_diagnostics", "calibration_diagnostics"]
        split_label = "fit"
    else:
        attr_candidates = ["test_diagnostics"]
        split_label = "test"

    diag = None
    for attr in attr_candidates:
        value = getattr(result, attr, None)
        if value is not None:
            diag = dict(value)
            break

    if diag is None:
        diag = {}
        df = getattr(result, "report_vs_pseudo", None)
        if df is not None:
            try:
                row = df[(df["Method"] == "Quad4FHE") & (df["Split"] == split_label)]
                if len(row) > 0:
                    diag["agreement"] = float(row.iloc[0]["Agreement"])
            except Exception:
                pass

    n_value = diag.get("n", diag.get("calib_n", diag.get("n_samples", n_expected)))
    agreement = diag.get("agreement", diag.get("calib_agreement", diag.get("fit_agreement")))
    mismatch = diag.get("mismatch_count", diag.get("calib_mismatch_count", diag.get("fit_mismatch_count")))

    if mismatch is None and agreement is not None and n_value is not None:
        mismatch = int(round((1.0 - float(agreement)) * int(n_value)))

    exact = diag.get("exact_preserved", diag.get("exact_preserved_on_calib", diag.get("fit_exact_preserved")))
    if exact is None and mismatch is not None:
        exact = bool(int(mismatch) == 0)

    return {
        "split": split_label,
        "n": to_jsonable(n_value),
        "agreement": to_jsonable(agreement),
        "mismatch_count": to_jsonable(mismatch),
        "exact_preserved": to_jsonable(exact),
    }


def quad_solver_diagnostics(result: Any) -> Dict[str, Any]:
    return to_jsonable(dict(getattr(result, "solver_diagnostics", None) or {}))


def build_quad4fhe_run_record(
    *,
    result: Any,
    key: str,
    hidden_dim: Optional[int],
    fit_n: Optional[int],
    test_n: Optional[int],
    pool_fraction: Optional[float] = None,
    rep_fit_size: Optional[int] = None,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """Build a JSON-serializable record for one Quad4FHE run."""
    fit_diag = quad_report_diagnostics(result, "fit", n_expected=fit_n)
    test_diag = quad_report_diagnostics(result, "test", n_expected=test_n)
    solver_diag = quad_solver_diagnostics(result)

    report_truth_test = dataframe_test_by_method(getattr(result, "report_vs_truth", None))
    report_pseudo_test = dataframe_test_by_method(getattr(result, "report_vs_pseudo", None))

    calib_agreement = fit_diag.get("agreement")
    test_agreement = test_diag.get("agreement")
    gap = None
    if calib_agreement is not None and test_agreement is not None:
        gap = float(calib_agreement) - float(test_agreement)

    # Common keys requested by reviewers. Keep all solver diagnostics too.
    common_solver_keys = [
        "num_pairwise_constraints",
        "min_pairwise_margin",
        "normalized_min_pairwise_margin",
        "slack_positive_count",
        "sum_slack",
        "mean_slack",
        "max_slack",
        "pairwise_slack_positive_count",
        "pairwise_sum_slack",
        "pairwise_mean_slack",
        "pairwise_max_slack",
        "selected_C",
        "soft_C_grid",
        "soft_trace",
        "selected_mu",
        "mu_grid",
        "mu_p",
        "mu_n",
    ]

    quad = {
        "alpha": getattr(result, "alpha", None),
        "beta": getattr(result, "beta", None),
        "eta": getattr(result, "eta", None),
        "threshold": getattr(result, "threshold", None),
        "zero_threshold_realized": getattr(result, "zero_threshold_realized", None),
        "method_used": getattr(result, "method_used", None),
        "hard_feasible": getattr(result, "feasible", None),
        "empirical_margin": getattr(result, "empirical_margin", None),
        "normalized_margin": getattr(result, "normalized_margin", None),
        "quant_decimals": getattr(result, "quant_decimals", None),
        "constraint_version": getattr(result, "constraint_version", None),
        "he_artifact_dir": str(getattr(result, "he_export_dir", None)) if getattr(result, "he_export_dir", None) else None,
        "calib_n": fit_diag.get("n"),
        "calib_agreement": calib_agreement,
        "calib_mismatch_count": fit_diag.get("mismatch_count"),
        "exact_preserved_on_calib": fit_diag.get("exact_preserved"),
        "test_n": test_diag.get("n"),
        "test_agreement": test_agreement,
        "test_mismatch_count": test_diag.get("mismatch_count"),
        "calib_test_agreement_gap": gap,
        "test_top1_acc": _metric_from_table(report_truth_test, "Quad4FHE", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Quad4FHE", "Top5", "Top-5"),
        "test_macro_f1": _metric_from_table(report_truth_test, "Quad4FHE", "MacroF1", "F1"),
        "solver_diagnostics": solver_diag,
    }
    for k in common_solver_keys:
        quad[k] = solver_diag.get(k)

    # Fill a few derived diagnostics that are convenient for tables.
    if quad.get("pairwise_mean_slack") is None:
        denom = quad.get("num_pairwise_constraints")
        num = quad.get("pairwise_sum_slack")
        try:
            if denom not in (None, 0) and num is not None:
                quad["pairwise_mean_slack"] = float(num) / float(denom)
        except Exception:
            pass
    if quad.get("selected_mu") is None:
        method = quad.get("method_used")
        try:
            if isinstance(method, str) and method.startswith("rch") and quad.get("mu_p") == quad.get("mu_n"):
                quad["selected_mu"] = quad.get("mu_p")
        except Exception:
            pass

    original = {
        "test_top1_acc": _metric_from_table(report_truth_test, "Original", "ACC", "Top1", "Top-1"),
        "test_top5_acc": _metric_from_table(report_truth_test, "Original", "Top5", "Top-5"),
    }

    record = {
        "key": key,
        "hidden_dim": hidden_dim,
        "pool_fraction": pool_fraction,
        "rep_fit_size": rep_fit_size,
        "original": original,
        "quad4fhe": quad,
        "report_vs_ground_truth_test": report_truth_test,
        "report_vs_pseudo_labels_test": report_pseudo_test,
        "report_vs_ground_truth_records": dataframe_records(getattr(result, "report_vs_truth", None)),
        "report_vs_pseudo_labels_records": dataframe_records(getattr(result, "report_vs_pseudo", None)),
    }
    if extra:
        record.update(to_jsonable(extra))
    return to_jsonable(record)


def make_experiment_payload(
    *,
    dataset: str,
    experiment: str,
    seed: int,
    dataset_info: Dict[str, Any],
    config: Dict[str, Any],
    source_script: Optional[str] = None,
    log_file: Optional[str | Path] = None,
) -> Dict[str, Any]:
    return {
        "dataset": dataset,
        "experiment": experiment,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "source_script": source_script,
        "log_file": str(log_file) if log_file is not None else None,
        "seed": seed,
        "dataset_info": to_jsonable(dataset_info),
        "config": to_jsonable(config),
        "runs": [],
    }


def write_combined_dataset_json(dataset: str, root_dir: str | Path = "results",
                                output_path: Optional[str | Path] = None) -> Optional[Path]:
    """Merge autosaved fulltrain/smallpool JSONs into one dataset-level JSON.

    This keeps compatibility with the older `<dataset>_results.json` convention.
    Missing halves are skipped, so it is safe to call after either notebook.
    """
    root = Path(root_dir) / dataset
    if output_path is None:
        output_path = root / f"{dataset}_results.json"
    output_path = Path(output_path)

    combined: Dict[str, Any] = {"dataset": dataset, "created_at": datetime.now().isoformat(timespec="seconds")}
    found = False
    for exp in ("fulltrain", "smallpool"):
        p = root / exp / f"{dataset}_{exp}_results.json"
        if not p.exists():
            continue
        with p.open("r", encoding="utf-8") as fh:
            block = json.load(fh)
        combined[exp] = {
            "source_json": str(p),
            "source_script": block.get("source_script"),
            "log_file": block.get("log_file"),
            "dataset_info": block.get("dataset_info", {}),
            "config": block.get("config", {}),
            "runs": block.get("runs", []),
        }
        if exp == "fulltrain":
            combined[exp]["cross_hidden_dim_summary"] = block.get("cross_hidden_dim_summary", [])
        if exp == "smallpool":
            combined[exp]["cross_pool_tables"] = block.get("cross_pool_tables", {})
            combined[exp]["meta_table"] = block.get("meta_table", [])
        found = True

    if not found:
        print(f"[autosave] no fulltrain/smallpool JSON found under {root}; combined JSON not written")
        return None
    return save_json(combined, output_path)


import traceback


In [2]:
import os
import gc
import sys
import copy
import time
import random
import hashlib
import warnings
import faulthandler
from contextlib import nullcontext
from pathlib import Path
from typing import Dict, List

faulthandler.enable()
warnings.filterwarnings("ignore")

os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from huggingface_hub import hf_hub_download, list_repo_files
    HAS_HF_HUB = True
except Exception:
    hf_hub_download = None
    list_repo_files = None
    HAS_HF_HUB = False

try:
    from transformers import AutoTokenizer, AutoModel
    HAS_TRANSFORMERS = True
except Exception:
    AutoTokenizer = None
    AutoModel = None
    HAS_TRANSFORMERS = False

import quad4fhe


# ============================================================
# Configuration
# ============================================================
SEED = 2026

# ---------- Banking77-specific ----------
# PolyAI/banking77 still ships a legacy Python loader script (banking77.py) which
# is no longer supported by datasets>=4.0.0. mteb/banking77 is the MTEB-maintained
# parquet mirror (identical content: 9,999 train + 3,080 test, 77 intents).
HF_DATASET_NAME = "mteb/banking77"
DATASET_NAME = "Qwen3_banking77"
EXPERIMENT_NAME = "smallpool"
NUM_CLASSES  = 77
# ----------------------------------------

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MAX_LENGTH = 128

HIDDEN_DIM = 256

# 4-way split fractions (must sum to 1.0)
TRAIN_SIZE     = 0.50
VAL_SIZE       = 0.10
TEST_SIZE      = 0.20
POOL_MAX_SIZE  = 0.20

# Fractions of TOTAL dataset used as rep_fit for each pool size.
# Banking77 total pool (train + test merged) ~ 13,079 samples. Per-fraction counts:
#   1%  ->   ~131 samples (~1.7/class for 77 classes) -- tight, may fall back
#                                                       to non-stratified sampling
#   5%  ->   ~654 samples (~8.5/class)
#  10%  ->  ~1308 samples (~17/class)
#  20%  ->  ~2616 samples (~34/class) -- uses the entire pool_max.
SURROGATE_FRACTIONS = [0.01, 0.05, 0.10, 0.20]

# Minimum pool size gate: need >= NUM_CLASSES for stratified split to be valid.
# For Banking77 (77 classes) at 1% = ~131 samples, we're just above this floor.
MIN_POOL_SAMPLES      = max(NUM_CLASSES, 50)
MIN_SAMPLES_PER_CLASS = 1         # warn (not skip) if any class < this

EPOCHS = 100
LR = 1e-3
BATCH_SIZE_TRAIN = 256
BATCH_SIZE_EVAL = 1024
WEIGHT_DECAY = 1e-5
PATIENCE = 10
PRINT_EVERY = 5

BATCH_SIZE_EMBED = 16
USE_AMP       = True
NORMALIZE_EMB = True

DEFAULT_INSTRUCTION = "Classify the user intent in this banking customer service query."

# --- Paths ---
FEATURE_CACHE_DIR = "./data"
OUTPUT_DIR = Path("..") / "results" / DATASET_NAME / EXPERIMENT_NAME
LOG_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_result.txt"
JSON_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_results.json"
META_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_{EXPERIMENT_NAME}_meta.csv"

BASELINES = ["square", "ls_poly_2", "ls_poly_3", "ls_poly_5", "ls_poly_7",
             "remez_2", "remez_3", "remez_5", "remez_7",
             "ola", "precise_a11"]

BANKING77_CLASS_NAMES: Dict[int, str] = {}


# ============================================================
# Utilities
# ============================================================
def separator(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)
    sys.stdout.flush()


def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


# ============================================================
# Banking77 loading: merge train + test into a single pool
# ============================================================
def _find_parquet_files(repo_id: str, split_name: str) -> List[str]:
    """Return repo-relative parquet file paths belonging to the given split."""
    if not HAS_HF_HUB:
        raise ImportError("huggingface_hub is required. `pip install -U huggingface_hub`")
    files = list_repo_files(repo_id, repo_type="dataset")
    matches = []
    split_lc = split_name.lower()
    for f in files:
        if not f.endswith(".parquet"):
            continue
        parts = f.lower().split("/")
        basename = parts[-1].removesuffix(".parquet") if hasattr(str, "removesuffix") \
                   else parts[-1][:-len(".parquet")]
        if split_lc in parts or basename == split_lc or basename.startswith(split_lc + "-"):
            matches.append(f)
    if not matches:
        raise FileNotFoundError(
            f"No parquet files matching split='{split_name}' in {repo_id}. "
            f"All files: {files}"
        )
    return sorted(matches)


def _load_parquet_split(repo_id: str, split_name: str) -> pd.DataFrame:
    paths = _find_parquet_files(repo_id, split_name)
    frames = []
    for fp in paths:
        local = hf_hub_download(repo_id=repo_id, filename=fp, repo_type="dataset")
        frames.append(pd.read_parquet(local))
    df = pd.concat(frames, ignore_index=True)
    print(f"  [{split_name}] loaded {len(df)} rows from {len(paths)} parquet shard(s)")
    return df


def _df_to_texts_labels(df: pd.DataFrame, split_tag: str):
    """Extract (texts, labels, name_map) from a Banking77 parquet DataFrame."""
    for col in ("text", "utt", "sentence"):
        if col in df.columns:
            text_col = col; break
    else:
        raise KeyError(f"Banking77 {split_tag}: no text column. Got {list(df.columns)}")
    for col in ("label", "intent", "labels"):
        if col in df.columns:
            label_col = col; break
    else:
        raise KeyError(f"Banking77 {split_tag}: no label column. Got {list(df.columns)}")

    texts = [str(x) for x in df[text_col].tolist()]
    raw = df[label_col].tolist()
    if len(raw) > 0 and isinstance(raw[0], str):
        unique = sorted({str(v) for v in raw})
        str_to_idx = {s: i for i, s in enumerate(unique)}
        labels = np.asarray([str_to_idx[str(v)] for v in raw], dtype=np.int64)
        name_map = {i: s for s, i in str_to_idx.items()}
    else:
        labels = np.asarray(raw, dtype=np.int64)
        name_map: Dict[int, str] = {}
        if "label_text" in df.columns:
            for lb, nm in zip(labels.tolist(), df["label_text"].tolist()):
                if lb not in name_map and nm is not None:
                    name_map[int(lb)] = str(nm)
    return texts, labels, name_map


def load_banking77_full_pool(dataset_name=HF_DATASET_NAME):
    """Load Banking77 train + test and concatenate into a single (texts, labels) pool."""
    global BANKING77_CLASS_NAMES

    df_train = _load_parquet_split(dataset_name, "train")
    df_test  = _load_parquet_split(dataset_name, "test")

    tr_texts, tr_labels, nm_tr = _df_to_texts_labels(df_train, "train")
    te_texts, te_labels, nm_te = _df_to_texts_labels(df_test,  "test")

    combined_names = dict(nm_tr)
    for k, v in nm_te.items():
        combined_names.setdefault(k, v)
    if combined_names:
        BANKING77_CLASS_NAMES = combined_names
        n = len(BANKING77_CLASS_NAMES)
        if n != NUM_CLASSES:
            print(f"  WARNING: NUM_CLASSES={NUM_CLASSES} but dataset yielded {n} class names. "
                  f"Update NUM_CLASSES to match.")
        print(f"  Loaded {n} Banking77 class names from records")

    all_texts  = tr_texts + te_texts
    all_labels = np.concatenate([tr_labels, te_labels], axis=0).astype(np.int64)
    print(f"  Merged pool: {len(all_texts)} = {len(tr_texts)} (train) + {len(te_texts)} (test)")
    return all_texts, all_labels


def split_4way_indices(y, train_size=TRAIN_SIZE, val_size=VAL_SIZE,
                        test_size=TEST_SIZE, pool_size=POOL_MAX_SIZE, seed=SEED):
    """Split into 4 disjoint subsets summing to 1.0: train / val / test / pool_max."""
    total = train_size + val_size + test_size + pool_size
    if abs(total - 1.0) > 1e-6:
        raise ValueError(f"Split fractions must sum to 1.0, got {total:.6f}")

    idx_all = np.arange(len(y))
    idx_rest, idx_test = train_test_split(idx_all, test_size=test_size, random_state=seed, stratify=y)
    pool_frac_of_rest = pool_size / (1.0 - test_size)
    idx_rest2, idx_pool_max = train_test_split(
        idx_rest, test_size=pool_frac_of_rest, random_state=seed + 1, stratify=y[idx_rest])
    val_frac_of_rest2 = val_size / (train_size + val_size)
    idx_train, idx_val = train_test_split(
        idx_rest2, test_size=val_frac_of_rest2, random_state=seed + 2, stratify=y[idx_rest2])

    return {
        "train":    np.sort(idx_train),
        "val":      np.sort(idx_val),
        "pool_max": np.sort(idx_pool_max),
        "test":     np.sort(idx_test),
    }


def subsample_pool(y_pool, fraction_of_total, total_size, seed):
    """Stratified subsample of size round(fraction*total) from the pool."""
    target = int(round(total_size * fraction_of_total))
    target = max(target, 1)
    n_pool = len(y_pool)
    if target >= n_pool:
        return np.arange(n_pool, dtype=np.int64)
    idx = np.arange(n_pool, dtype=np.int64)
    n_classes_present = len(np.unique(y_pool))
    can_stratify = target >= n_classes_present
    if can_stratify:
        try:
            sub_idx, _ = train_test_split(idx, train_size=target, random_state=seed, stratify=y_pool)
            return np.sort(sub_idx)
        except ValueError as e:
            print(f"  [subsample] stratified split failed ({e}); falling back to non-stratified.")
    print(f"  [subsample] Non-stratified sampling: target={target} < classes={n_classes_present}.")
    rng = np.random.default_rng(seed)
    sub_idx = rng.choice(idx, size=target, replace=False)
    return np.sort(sub_idx)


# ============================================================
# Qwen3 embedding extraction
# ============================================================
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f"Instruct: {task_description}\nQuery:{query}"


def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    seq_lens = attention_mask.sum(dim=1) - 1
    batch = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch, device=last_hidden_states.device), seq_lens]


def _autocast_context(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


@torch.no_grad()
def embed_texts_qwen3(texts, model_name, device, batch_size=BATCH_SIZE_EMBED,
                       max_length=MAX_LENGTH, instruction=DEFAULT_INSTRUCTION,
                       normalize=NORMALIZE_EMB, amp=USE_AMP):
    if not HAS_TRANSFORMERS:
        raise ImportError("transformers>=4.51.0 is required.")

    print(f"Embedding {len(texts)} texts with {model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {}
    if device.type == "cuda":
        model_kwargs["torch_dtype"] = torch.float16
    model = AutoModel.from_pretrained(model_name, **model_kwargs).to(device)
    model.eval()

    use_amp = amp and device.type == "cuda"
    outputs_all = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    for b, start in enumerate(range(0, len(texts), batch_size), start=1):
        batch_texts = texts[start:start + batch_size]
        if instruction:
            batch_texts = [get_detailed_instruct(instruction, t) for t in batch_texts]
        batch = tokenizer(batch_texts, padding=True, truncation=True,
                           max_length=max_length, return_tensors="pt")
        batch = {k: v.to(device) for k, v in batch.items()}
        with _autocast_context(device, use_amp):
            out = model(**batch)
            emb = last_token_pool(out.last_hidden_state, batch["attention_mask"])
            if normalize:
                emb = F.normalize(emb, p=2, dim=1)
        outputs_all.append(emb.detach().cpu().float().numpy())
        if b == 1 or b % 50 == 0 or b == n_batches:
            print(f"  Embed batch {b:>5d}/{n_batches}")

    del model, tokenizer
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("  Qwen3 backbone released from memory.")
    return np.concatenate(outputs_all, axis=0).astype(np.float32)


def feature_cache_path(cache_dir, model_name, max_length, instruction):
    os.makedirs(cache_dir, exist_ok=True)
    digest = hashlib.md5((str(max_length) + "|" + str(instruction or "")).encode("utf-8")).hexdigest()[:10]
    model_tag = model_name.split("/")[-1].replace("-", "_")
    return os.path.join(cache_dir, f"{model_tag}_banking77_pool_{digest}_features.npz")


def ensure_qwen3_feature_cache_pool(texts, labels, model_name, cache_path, device):
    """Compute (or load) Qwen3 embeddings for the FULL merged pool."""
    if os.path.exists(cache_path):
        print(f"Loading cached features: {cache_path}")
        pack = np.load(cache_path, allow_pickle=False)
        if "features" in pack and "labels" in pack:
            feats = pack["features"].astype(np.float32)
            labels_cached = pack["labels"].astype(np.int64)
            if feats.shape[0] == len(labels) and np.array_equal(labels_cached, labels):
                print(f"  Cache OK: shape={feats.shape}")
                return feats, labels_cached
            print("  Cache size/labels mismatch -> recomputing features...")
        else:
            print("  Cache missing required arrays -> recomputing features...")

    t0 = time.perf_counter()
    feats = embed_texts_qwen3(texts, model_name, device)
    print(f"Pool embedding time: {time.perf_counter() - t0:.1f}s, feats.shape={feats.shape}")

    np.savez(cache_path, features=feats, labels=labels)
    print(f"Saved feature cache: {cache_path}")
    return feats, labels


# ============================================================
# Classifier head + training
# ============================================================
class FeatureClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=1e-5):
        self.patience = patience; self.min_delta = min_delta
        self.best = float("inf"); self.counter = 0; self.best_state = None

    def step(self, loss, model):
        if loss < self.best - self.min_delta:
            self.best = loss; self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)


def train_head(X_tr, y_tr, X_va, y_va, input_dim, hidden_dim, num_classes, device):
    model = FeatureClassifier(input_dim, hidden_dim, num_classes).to(device)
    tr_loader = DataLoader(TensorDataset(torch.tensor(X_tr, dtype=torch.float32),
                                          torch.tensor(y_tr, dtype=torch.long)),
                           batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    va_loader = DataLoader(TensorDataset(torch.tensor(X_va, dtype=torch.float32),
                                          torch.tensor(y_va, dtype=torch.long)),
                           batch_size=BATCH_SIZE_EVAL, shuffle=False)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ce = nn.CrossEntropyLoss()
    stopper = EarlyStopping()
    amp_enabled = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler(device="cuda", enabled=amp_enabled)

    print(f"  Architecture: Frozen Qwen3-Embedding-0.6B({input_dim}) -> "
          f"Linear({input_dim}->{hidden_dim}) -> ReLU -> Linear({hidden_dim}->{num_classes})")
    print(f"  Device: {device}, AMP: {amp_enabled}")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, total_n = 0.0, 0
        for xb, yb in tr_loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                logits = model(xb); loss = ce(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total_loss += loss.item() * xb.size(0); total_n += xb.size(0)
        train_loss = total_loss / max(total_n, 1)

        model.eval()
        v_loss, v_n = 0.0, 0
        with torch.no_grad():
            for xb, yb in va_loader:
                xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    logits = model(xb); loss = ce(logits, yb)
                v_loss += loss.item() * xb.size(0); v_n += xb.size(0)
        val_loss = v_loss / max(v_n, 1)

        if epoch == 1 or epoch % PRINT_EVERY == 0 or epoch == EPOCHS:
            print(f"    Epoch {epoch:>3d}/{EPOCHS} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if stopper.step(val_loss, model):
            print(f"    Early stopping at epoch {epoch}, best_val_loss={stopper.best:.6f}")
            break

    stopper.restore(model)
    model.cpu().eval()
    return model


# ============================================================
# Main
# ============================================================
def main():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not HAS_TRANSFORMERS:
        raise ImportError("This script requires transformers>=4.51.0")
    if not HAS_HF_HUB:
        raise ImportError("This script requires huggingface_hub. `pip install -U huggingface_hub`")

    # ---------------- Step 1: Load Banking77 (merged pool) + 4-way split ----------------
    separator("Step 1: Load Banking77 (merged pool) and create 4-way split "
              f"(train {TRAIN_SIZE*100:.0f}% / val {VAL_SIZE*100:.0f}% / "
              f"test {TEST_SIZE*100:.0f}% / pool_max {POOL_MAX_SIZE*100:.0f}%)")
    print(f"Dataset             : {DATASET_NAME}")
    print(f"HF dataset          : {HF_DATASET_NAME}")
    print(f"Num classes         : {NUM_CLASSES}")
    print(f"Model               : {MODEL_NAME}")
    print(f"HIDDEN_DIM          = {HIDDEN_DIM}")
    print(f"SURROGATE_FRACTIONS = {SURROGATE_FRACTIONS}")

    all_texts, all_labels = load_banking77_full_pool(HF_DATASET_NAME)
    total_size = len(all_texts)
    print(f"\nTotal pool size: {total_size}, classes present: "
          f"{len(np.unique(all_labels))}/{NUM_CLASSES}")

    split_idx = split_4way_indices(all_labels)
    for key in ["train", "val", "pool_max", "test"]:
        idx = split_idx[key]
        pct = len(idx) / total_size * 100.0
        vals = np.unique(all_labels[idx])
        print(f"  {key:10s}: n={len(idx):6d} ({pct:5.2f}%), classes_present={len(vals)}/{NUM_CLASSES}")

    print("\n  Pool subset sizes (per SURROGATE_FRACTIONS):")
    for f in SURROGATE_FRACTIONS:
        n_target = int(round(total_size * f))
        n_actual = min(n_target, len(split_idx["pool_max"]))
        per_class = n_target / NUM_CLASSES
        note = ""
        if n_target < NUM_CLASSES:
            note = f"  [!] target < {NUM_CLASSES} classes -> non-stratified fallback"
        print(f"    fraction={f*100:5.1f}% -> target n={n_target:6d} "
              f"(~{per_class:.1f}/class), actual n={n_actual:6d}{note}")

    # ---------------- Step 2: Qwen3 embeddings for full pool ----------------
    separator("Step 2: Extract / load frozen Qwen3 embeddings (full pool)")
    cache_path = feature_cache_path(FEATURE_CACHE_DIR, MODEL_NAME, MAX_LENGTH, DEFAULT_INSTRUCTION)
    feats_all, labels_all = ensure_qwen3_feature_cache_pool(
        all_texts, all_labels, MODEL_NAME, cache_path, device)
    print(f"  Feature matrix shape: {feats_all.shape}")

    X_train    = feats_all[split_idx["train"]];    y_train    = labels_all[split_idx["train"]]
    X_val      = feats_all[split_idx["val"]];      y_val      = labels_all[split_idx["val"]]
    X_pool_max = feats_all[split_idx["pool_max"]]; y_pool_max = labels_all[split_idx["pool_max"]]
    X_test     = feats_all[split_idx["test"]];     y_test     = labels_all[split_idx["test"]]

    # Fit scaler on train only (no leakage)
    scaler = StandardScaler().fit(X_train)
    X_train_std    = scaler.transform(X_train).astype(np.float64)
    X_val_std      = scaler.transform(X_val).astype(np.float64)
    X_pool_max_std = scaler.transform(X_pool_max).astype(np.float64)
    X_test_std     = scaler.transform(X_test).astype(np.float64)
    input_dim = X_train_std.shape[1]
    print(f"  Qwen3 embedding dim = {input_dim}")

    # ---------------- Step 3: Train head once ----------------
    separator(f"Step 3: Train classifier head (hidden_dim={HIDDEN_DIM})")
    set_seed(SEED)
    model = train_head(X_train_std, y_train, X_val_std, y_val,
                       input_dim, HIDDEN_DIM, NUM_CLASSES, device)

    with torch.no_grad():
        orig_test_logits = model(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
    orig_top1 = float(np.mean(orig_test_logits.argmax(axis=1) == y_test))
    print(f"\n  Original head test top-1 ACC = {orig_top1:.4f}")

    # ---------------- Step 4: Loop over pool fractions ----------------
    separator("Step 4: quad4fhe.replace() per pool fraction")
    all_pool_results = {}

    all_pool_fit_sizes = {}
    all_pool_skips = {}
    payload = make_experiment_payload(
        dataset=DATASET_NAME,
        experiment=EXPERIMENT_NAME,
        seed=SEED,
        dataset_info={
            "input_dim": int(input_dim),
            "num_classes": int(NUM_CLASSES),
            "splits": {
                "source": "fixed_stratified_split_from_official_train_test",
                "train": int(len(X_train_std)),
                "val": int(len(X_val_std)),
                "pool_max": int(len(X_pool_max_std)),
                "test": int(len(X_test_std)),
            },
            "hf_dataset": HF_DATASET_NAME,
            "embedding_model": MODEL_NAME,
            "embedding_normalized": NORMALIZE_EMB,
            "feature_cache_dir": FEATURE_CACHE_DIR,
            "frozen_backbone": True,
            "encrypted_component": "classification_head_only",
        },
        config={
            "hidden_dim": HIDDEN_DIM,
            "epochs": EPOCHS,
            "lr": LR,
            "batch_size_train": BATCH_SIZE_TRAIN,
            "batch_size_eval": BATCH_SIZE_EVAL,
            "batch_size_embed": BATCH_SIZE_EMBED,
            "weight_decay": WEIGHT_DECAY,
            "patience": PATIENCE,
            "max_length": MAX_LENGTH,
            "use_amp": USE_AMP,
            "normalize_emb": NORMALIZE_EMB,
            "default_instruction": DEFAULT_INSTRUCTION,
            "default_configs": globals().get("DEFAULT_CONFIGS"),
            "massive_label_field": globals().get("MASSIVE_LABEL_FIELD"),
            "surrogate_fractions": SURROGATE_FRACTIONS,
            "train_size": globals().get("TRAIN_SIZE"),
            "val_size": globals().get("VAL_SIZE"),
            "test_size": globals().get("TEST_SIZE"),
            "pool_max_size": globals().get("POOL_MAX_SIZE"),
            "min_pool_samples": globals().get("MIN_POOL_SAMPLES"),
            "min_samples_per_class": globals().get("MIN_SAMPLES_PER_CLASS"),
            "baselines": BASELINES,
            "split_seed": SEED,
            "randomness": "single fixed stratified split; not repeated random subsampling",
        },
        source_script=f"{DATASET_NAME}_{EXPERIMENT_NAME}_autosave.ipynb",
        log_file=LOG_PATH,
    )

    for fraction in SURROGATE_FRACTIONS:
        print("\n" + "-" * 78)
        print(f"POOL = {fraction*100:.0f}%")
        print("-" * 78)

        sub_idx = subsample_pool(y_pool_max, fraction, total_size, seed=SEED)
        X_rep_fit = X_pool_max_std[sub_idx]
        y_rep_fit = y_pool_max[sub_idx]

        if len(X_rep_fit) < MIN_POOL_SAMPLES:
            print(f"  [SKIPPED] rep_fit too small (n={len(X_rep_fit)} < {MIN_POOL_SAMPLES})")
            all_pool_results[fraction] = None
            all_pool_skips[fraction] = "skipped_by_pool_guard"
            continue

        counts = np.bincount(y_rep_fit, minlength=NUM_CLASSES)
        missing = [i for i, c in enumerate(counts) if c < MIN_SAMPLES_PER_CLASS]
        if missing:
            print(f"  [WARN] {len(missing)} classes have < {MIN_SAMPLES_PER_CLASS} samples in rep_fit "
                  f"(first 10: {missing[:10]}{'...' if len(missing)>10 else ''})")

        print(f"  rep_fit: n={len(X_rep_fit)}, "
              f"per-class counts: min={counts.min()}, max={counts.max()}, mean={counts.mean():.2f}, "
              f"classes present={np.sum(counts > 0)}/{NUM_CLASSES}")

        result = quad4fhe.replace(
            task              = "multiclass",
            model             = model,
            hidden_layer      = "fc1",
            output_layer      = "fc2",
            activation        = "relu",
            fit_data          = (X_rep_fit, y_rep_fit),
            test_data         = (X_test_std, y_test),
            baselines         = BASELINES,
            export_he_dir     = f"he_artifacts_qwen3_banking77_pool_{int(fraction*100):02d}",
            use_cutting_plane = True,
            use_osqp_warmstart= False,
            verbose           = True,
            seed              = SEED,
        )
        all_pool_results[fraction] = result

        all_pool_fit_sizes[fraction] = len(X_rep_fit)
        fit_diag = quad_report_diagnostics(result, "fit", n_expected=len(X_rep_fit))
        test_diag = quad_report_diagnostics(result, "test", n_expected=len(X_test_std))
        payload["runs"].append(build_quad4fhe_run_record(
            result=result,
            key=f"pool={int(fraction*100):02d}%",
            hidden_dim=HIDDEN_DIM,
            fit_n=len(X_rep_fit),
            test_n=len(X_test_std),
            pool_fraction=fraction,
            rep_fit_size=len(X_rep_fit),
            extra={"pool_label": f"{int(fraction*100):02d}%"},
        ))
        save_json(payload, JSON_PATH)
        print(f"  calib agreement={fit_diag.get('agreement')} "
              f"mismatches={fit_diag.get('mismatch_count')} "
              f"exact_preserved={fit_diag.get('exact_preserved')}")
        print(f"  test agreement={test_diag.get('agreement')} "
              f"mismatches={test_diag.get('mismatch_count')}")

        rm = result.replaced_model.eval()
        with torch.no_grad():
            logits_quad = rm(torch.tensor(X_test_std, dtype=torch.float32)).cpu().numpy()
        quad_top1 = float(np.mean(logits_quad.argmax(axis=1) == y_test))

        print(f"  method_used={result.method_used}, "
              f"hard_feasible={result.feasible}, "
              f"quant_decimals={result.quant_decimals}, "
              f"quad_top1={quad_top1:.4f}")

        # Save per-fraction report
        frac_dir = os.path.join(OUTPUT_DIR, f"pool_{int(fraction*100):02d}")
        os.makedirs(frac_dir, exist_ok=True)
        if result.report_vs_truth is not None:
            result.report_vs_truth.to_csv(os.path.join(frac_dir, "report_vs_truth.csv"), index=False)

    # ---------------- Step 5: Cross-pool comparison ----------------
    separator("Step 5: Cross-pool comparison")

    pool_labels = [f"{int(f*100):02d}%" for f in SURROGATE_FRACTIONS]

    first = next((r for r in all_pool_results.values() if r is not None), None)
    if first is not None and first.report_vs_truth is not None:
        available_cols = list(first.report_vs_truth.columns)
    else:
        available_cols = []
    METRIC_KEYS = [c for c in ["ACC", "MacroF1", "MacroPrec", "MacroRec", "AUC_ovr", "Agreement"]
                   if c in available_cols]

    method_order = ["Original", "Quad4FHE"] + BASELINES

    def extract_test_row(res, method):
        if res is None or res.report_vs_truth is None: return None
        df = res.report_vs_truth
        r = df[(df["Method"] == method) & (df["Split"] == "test")]
        return r.iloc[0].to_dict() if len(r) > 0 else None

    def _build_matrix(metric):
        data = {}
        for f, lbl in zip(SURROGATE_FRACTIONS, pool_labels):
            col = {}
            for m in method_order:
                row = extract_test_row(all_pool_results[f], m)
                col[m] = row[metric] if row is not None and metric in row else float("nan")
            data[f"pool={lbl}"] = col
        return pd.DataFrame(data)

    cross_pool_tables = {}
    for metric in METRIC_KEYS:
        title = ("Test Agreement (with original model / pseudo-labels)"
                 if metric == "Agreement"
                 else f"Test {metric} (vs TRUE labels)")
        print(f"\n--- Table: {title} ---")
        df_m = _build_matrix(metric)
        cross_pool_tables[metric] = df_m.to_dict()
        with pd.option_context("display.float_format", "{:.4f}".format):
            print(df_m.to_string())

    print("\n--- Meta Table ---")
    meta_rows = []
    for f, lbl in zip(SURROGATE_FRACTIONS, pool_labels):
        r = all_pool_results[f]
        if r is None:
            row = {
                "pool": lbl,
                "status": "SKIPPED",
                "skip_reason": all_pool_skips.get(f, "skipped_by_pool_guard"),
                "pool_fraction": f,
                "hidden_dim": HIDDEN_DIM,
            }
            meta_rows.append(row)
            payload["runs"].append({
                "key": f"pool={lbl}",
                "hidden_dim": HIDDEN_DIM,
                "pool_fraction": f,
                "status": "skipped",
                "skip_reason": row["skip_reason"],
                "quad4fhe": {"status": "skipped", "skip_reason": row["skip_reason"]},
            })
            continue
        fit_diag = quad_report_diagnostics(r, "fit", n_expected=all_pool_fit_sizes.get(f))
        test_diag = quad_report_diagnostics(r, "test", n_expected=len(X_test_std))
        solver_diag = quad_solver_diagnostics(r)
        row = {
            "pool":             lbl,
            "pool_fraction":    f,
            "hidden_dim":       HIDDEN_DIM,
            "method_used":      r.method_used,
            "hard_feasible":    r.feasible,
            "empirical_margin": round(r.empirical_margin, 6),
            "norm_margin":      round(r.normalized_margin, 6),
            "quant_decimals":   r.quant_decimals,
            "calib_n":          fit_diag.get("n"),
            "calib_agreement":  fit_diag.get("agreement"),
            "calib_mismatch":   fit_diag.get("mismatch_count"),
            "exact_preserved":  fit_diag.get("exact_preserved"),
            "test_agreement":   test_diag.get("agreement"),
            "test_mismatch":    test_diag.get("mismatch_count"),
            "calib_test_gap":   (None if fit_diag.get("agreement") is None or test_diag.get("agreement") is None
                                  else fit_diag.get("agreement") - test_diag.get("agreement")),
            "num_pairwise_constraints": solver_diag.get("num_pairwise_constraints"),
            "min_pairwise_margin": solver_diag.get("min_pairwise_margin"),
            "normalized_min_pairwise_margin": solver_diag.get("normalized_min_pairwise_margin"),
            "slack_pos":        solver_diag.get("slack_positive_count"),
            "sum_slack":        solver_diag.get("sum_slack"),
            "mean_slack":       solver_diag.get("mean_slack"),
            "max_slack":        solver_diag.get("max_slack"),
            "pairwise_slack_pos": solver_diag.get("pairwise_slack_positive_count"),
            "pairwise_sum_slack": solver_diag.get("pairwise_sum_slack"),
            "pairwise_mean_slack": (None if solver_diag.get("pairwise_sum_slack") is None or solver_diag.get("num_pairwise_constraints") in (None, 0)
                                     else solver_diag.get("pairwise_sum_slack") / solver_diag.get("num_pairwise_constraints")),
            "pairwise_max_slack": solver_diag.get("pairwise_max_slack"),
            "selected_C":       solver_diag.get("selected_C"),
            "constraint_version": getattr(r, "constraint_version", None),
            "he_export_dir":    os.path.basename(r.he_export_dir) if r.he_export_dir else None,
        }
        meta_rows.append(row)
    df_meta = pd.DataFrame(meta_rows)
    print(df_meta.to_string(index=False))
    payload["cross_pool_tables"] = cross_pool_tables
    payload["meta_table"] = meta_rows
    save_json(payload, JSON_PATH)
    save_csv(meta_rows, META_CSV_PATH)
    write_combined_dataset_json(DATASET_NAME, root_dir=OUTPUT_DIR.parent.parent)

    separator("Done")


if __name__ == "__main__":
    try:
        with tee_stdout_stderr(LOG_PATH):
            main()
    except Exception:
        traceback.print_exc()
        sys.stdout.flush(); sys.stderr.flush()
        raise


[autosave] stdout/stderr log -> ../results/Qwen3_banking77/smallpool/Qwen3_banking77_smallpool_result.txt

Step 1: Load Banking77 (merged pool) and create 4-way split (train 50% / val 10% / test 20% / pool_max 20%)
Dataset             : Qwen3_banking77
HF dataset          : mteb/banking77
Num classes         : 77
Model               : Qwen/Qwen3-Embedding-0.6B
HIDDEN_DIM          = 256
SURROGATE_FRACTIONS = [0.01, 0.05, 0.1, 0.2]
  [train] loaded 9993 rows from 1 parquet shard(s)
  [test] loaded 3076 rows from 1 parquet shard(s)
  Loaded 77 Banking77 class names from records
  Merged pool: 13069 = 9993 (train) + 3076 (test)

Total pool size: 13069, classes present: 77/77
  train     : n=  6534 (50.00%), classes_present=77/77
  val       : n=  1307 (10.00%), classes_present=77/77
  pool_max  : n=  2614 (20.00%), classes_present=77/77
  test      : n=  2614 (20.00%), classes_present=77/77

  Pool subset sizes (per SURROGATE_FRACTIONS):
    fraction=  1.0% -> target n=   131 (~1.7/class),

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

  Embed batch     1/817
  Embed batch    50/817
  Embed batch   100/817
  Embed batch   150/817
  Embed batch   200/817
  Embed batch   250/817
  Embed batch   300/817
  Embed batch   350/817
  Embed batch   400/817
  Embed batch   450/817
  Embed batch   500/817
  Embed batch   550/817
  Embed batch   600/817
  Embed batch   650/817
  Embed batch   700/817
  Embed batch   750/817
  Embed batch   800/817
  Embed batch   817/817
  Qwen3 backbone released from memory.
Pool embedding time: 24.6s, feats.shape=(13069, 1024)
Saved feature cache: ./data/Qwen3_Embedding_0.6B_banking77_pool_c4d2f77533_features.npz
  Feature matrix shape: (13069, 1024)
  Qwen3 embedding dim = 1024

Step 3: Train classifier head (hidden_dim=256)
  Architecture: Frozen Qwen3-Embedding-0.6B(1024) -> Linear(1024->256) -> ReLU -> Linear(256->77)
  Device: cuda, AMP: True
    Epoch   1/100 | train_loss=1.930136 | val_loss=0.652304
    Epoch   5/100 | train_loss=0.186941 | val_loss=0.366433
    Epoch  10/100 | train_lo